# Demo 2 — MLflow RAG Evaluation: llama3.2 vs mistral

Demonstrates `mlflow.evaluate()` for RAG quality metrics, comparing two Ollama models
routed through the MLflow AI Gateway.

| Metric | What it measures |
|---|---|
| **faithfulness** | Is the answer grounded in the retrieved context? |
| **answer_relevance** | Does the answer address the question? |
| **context_relevance** | Is the retrieved context relevant to the question? |

**Judge LLM:** mistral (7B) via Ollama OpenAI-compatible endpoint  
**Answer generators:** `llama-chat` (llama3.2 3B) vs `mistral-chat` (mistral 7B)

**Prerequisites:** `docker compose up -d` in `demo2-mlflow-gateway/` — gateway healthy, models pulled.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

GATEWAY_URI    = os.getenv("MLFLOW_GATEWAY_URI", "http://localhost:5000")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL",   "http://localhost:11434")

os.environ["MLFLOW_GATEWAY_URI"] = GATEWAY_URI

print(f"Gateway : {GATEWAY_URI}")
print(f"Ollama  : {OLLAMA_BASE_URL}")

## 1. Build a RAG Evaluation Dataset

Each row needs: question, retrieved context, generated answer, and optionally a ground truth answer.

In [ ]:
import pandas as pd
from mlflow.deployments import get_deploy_client

client = get_deploy_client(GATEWAY_URI)

# Tax policy Q&A with ground truth and relevant context excerpts
eval_data = [
    {
        "question": "What is the standard deduction for a single filer in 2024?",
        "context": "For tax year 2024, the standard deduction amounts are: Single or Married Filing Separately: $14,600. Married Filing Jointly or Qualifying Surviving Spouse: $29,200. Head of Household: $21,900.",
        "ground_truth": "The standard deduction for a single filer in 2024 is $14,600.",
    },
    {
        "question": "What is the long-term capital gains rate for a single filer earning $100,000?",
        "context": "2024 Long-Term Capital Gains Tax Rates: 0%: Taxable income up to $47,025 (single). 15%: Taxable income from $47,026 to $518,900 (single). 20%: Taxable income above $518,900 (single).",
        "ground_truth": "A single filer earning $100,000 pays 15% on long-term capital gains.",
    },
    {
        "question": "What is the 401(k) employee contribution limit for 2024?",
        "context": "401(k), 403(b), AND 457(b) PLANS: Employee Contribution Limit: $23,000. Catch-Up Contribution (age 50 or older): Additional $7,500 (total $30,500).",
        "ground_truth": "The 401(k) employee contribution limit for 2024 is $23,000, or $30,500 with catch-up contributions for those 50 and older.",
    },
    {
        "question": "What is the Roth IRA income phase-out range for single filers in 2024?",
        "context": "ROTH IRA: Income phase-out ranges for contributions: Single/Head of Household: $146,000 – $161,000 MAGI.",
        "ground_truth": "The Roth IRA income phase-out range for single filers in 2024 is $146,000 to $161,000 MAGI.",
    },
    {
        "question": "Can I deduct mortgage interest on a second home?",
        "context": "Mortgage Interest: Interest on loans secured by your primary or secondary residence is deductible on up to $750,000 of acquisition debt (for loans originated after December 15, 2017).",
        "ground_truth": "Yes, mortgage interest on a secondary residence is deductible on up to $750,000 of acquisition debt for loans originated after December 15, 2017.",
    },
]

# Generate answers using llama-chat route (llama3.2 3B)
print("Generating answers via llama-chat route (llama3.2)...")
for row in eval_data:
    prompt = f"Answer the following tax question based only on the provided context.\n\nContext: {row['context']}\n\nQuestion: {row['question']}\n\nAnswer concisely."
    resp = client.predict(
        endpoint="llama-chat",
        inputs={"messages": [{"role": "user", "content": prompt}]},
    )
    row["answer"] = resp["choices"][0]["message"]["content"]
    print(f"  Q: {row['question'][:60]}...")
    print(f"  A: {row['answer'][:100]}\n")

eval_df = pd.DataFrame(eval_data)
print(f"Dataset ready: {len(eval_df)} rows")

## 2. MLflow Experiment Setup

Log RAG evaluation runs to an MLflow experiment for tracking and comparison.

In [ ]:
import mlflow

# Use local file-based tracking (no MLflow server needed)
mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("tax-rag-evaluation")

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment: tax-rag-evaluation")

## 3. RAG Evaluation — llama3.2 Answers, mistral as Judge

`mlflow.evaluate()` calls the judge LLM to score each row. We use **mistral** (7B via Ollama's
OpenAI-compatible endpoint) as the judge — a stronger model evaluating the smaller model's output.

In [ ]:
from mlflow.metrics.genai import faithfulness, answer_relevance, relevance

# Ollama exposes an OpenAI-compatible endpoint — point mlflow judge at it
OLLAMA_V1 = f"{OLLAMA_BASE_URL}/v1"
JUDGE_MODEL = "mistral"   # 7B model judges the 3B model's answers

with mlflow.start_run(run_name="llama32-answers-mistral-judge"):
    mlflow.log_param("rag_model",      "llama3.2")
    mlflow.log_param("judge_model",    JUDGE_MODEL)
    mlflow.log_param("gateway_route",  "llama-chat")

    results = mlflow.evaluate(
        data=eval_df,
        targets="ground_truth",
        predictions="answer",
        model_type="question-answering",
        extra_metrics=[
            faithfulness(    model=f"openai:/{JUDGE_MODEL}"),
            answer_relevance(model=f"openai:/{JUDGE_MODEL}"),
            relevance(       model=f"openai:/{JUDGE_MODEL}"),
        ],
        evaluator_config={
            "col_mapping": {"inputs": "question", "context": "context"},
            "openai_api_key":  "ollama",   # Ollama ignores the key
            "openai_api_base": OLLAMA_V1,
        },
    )

print("Evaluation complete.")
print("\nAggregate metrics (llama3.2 answers):")
for k, v in results.metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

## 4. Compare: mistral as Answer Generator

Swap the route from `llama-chat` → `mistral-chat`. Same eval harness, same judge.
This is how the gateway enables apples-to-apples model comparisons.

In [ ]:
import time

# Generate answers using mistral-chat route (mistral 7B)
print("Generating answers via mistral-chat route...")
mistral_rows = []
for row in eval_data:
    prompt = f"Answer the following tax question based only on the provided context.\n\nContext: {row['context']}\n\nQuestion: {row['question']}\n\nAnswer concisely."
    t0 = time.time()
    try:
        resp = client.predict(
            endpoint="mistral-chat",
            inputs={"messages": [{"role": "user", "content": prompt}]},
        )
        answer = resp["choices"][0]["message"]["content"]
    except Exception as e:
        answer = f"ERROR: {e}"
    elapsed = time.time() - t0
    mistral_rows.append({**row, "answer": answer, "latency_s": elapsed})
    print(f"  [{elapsed:.1f}s] {row['question'][:50]}...")

mistral_df = pd.DataFrame(mistral_rows)
print(f"\nmistral mean latency: {mistral_df['latency_s'].mean():.1f}s")

In [ ]:
with mlflow.start_run(run_name="mistral-answers-mistral-judge"):
    mlflow.log_param("rag_model",      "mistral")
    mlflow.log_param("judge_model",    JUDGE_MODEL)
    mlflow.log_param("gateway_route",  "mistral-chat")
    mlflow.log_metric("mean_latency_s", mistral_df["latency_s"].mean())

    mistral_results = mlflow.evaluate(
        data=mistral_df[["question", "context", "answer", "ground_truth"]],
        targets="ground_truth",
        predictions="answer",
        model_type="question-answering",
        extra_metrics=[
            faithfulness(    model=f"openai:/{JUDGE_MODEL}"),
            answer_relevance(model=f"openai:/{JUDGE_MODEL}"),
            relevance(       model=f"openai:/{JUDGE_MODEL}"),
        ],
        evaluator_config={
            "col_mapping": {"inputs": "question", "context": "context"},
            "openai_api_key":  "ollama",
            "openai_api_base": OLLAMA_V1,
        },
    )

print("mistral evaluation complete.")
print("\nAggregate metrics (mistral answers):")
for k, v in mistral_results.metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

## 5. Per-Row Results Comparison

In [ ]:
llama_per_row   = results.tables["eval_results_table"]
mistral_per_row = mistral_results.tables["eval_results_table"]

comparison = pd.DataFrame({
    "question":                  eval_df["question"],
    "llama32_faithfulness":      llama_per_row.get("faithfulness/v1/score"),
    "mistral_faithfulness":      mistral_per_row.get("faithfulness/v1/score"),
    "llama32_answer_relevance":  llama_per_row.get("answer_relevance/v1/score"),
    "mistral_answer_relevance":  mistral_per_row.get("answer_relevance/v1/score"),
})

print("=== llama3.2 (3B) vs mistral (7B) RAG Quality ===")
comparison

## 6. View in MLflow UI

All runs are logged locally. Launch the MLflow UI to compare runs visually:

```bash
cd demo2-mlflow-gateway
source .venv/bin/activate
mlflow ui --port 5001
```

Open http://localhost:5001 → experiment **tax-rag-evaluation** → compare the two runs side-by-side.

You'll see:
- Parameter comparison: which model/route was used
- Metric comparison: faithfulness, answer_relevance, relevance scores
- Per-row eval table with explanations from the judge LLM